In [53]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from qiskit_aer import Aer
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.utils.loss_functions import CrossEntropyLoss
from qiskit.algorithms.optimizers import COBYLA

In [54]:
df = pd.read_csv("C:/Users/B_Jayanth/OneDrive/Desktop/QML/src/cp2_feature_engineering/data/processed/checkpoint2/final_dataset_all_columns_cam1.csv")
df.head()

,camera_id,window_id,start_time,end_time,density,avg_velocity,velocity_variance,label
0,cam1,0,0.0,5.0,5,130.802768,45369.467076,1
1,cam1,1,5.0,10.0,6,183.054328,95381.534089,1
2,cam1,2,10.0,15.0,2,120.778352,29826.601906,1
3,cam1,3,15.0,20.0,6,126.183687,60297.368373,1
4,cam1,4,20.0,25.0,7,243.151864,127209.052544,1


In [55]:
df.shape

(424, 8)

In [56]:
df["label"].value_counts()

label
1    338
0     55
2     31
Name: count, dtype: int64

In [57]:
X = df[["density", "avg_velocity", "velocity_variance"]]
y = df["label"]

In [58]:
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [60]:
feature_map = ZZFeatureMap(feature_dimension=3, reps=2)

In [61]:
ansatz = RealAmplitudes(num_qubits=3, reps=4)

In [62]:
optimizer = COBYLA(maxiter=300)

In [63]:
backend = Aer.get_backend("qasm_simulator")

In [64]:
from qiskit.primitives import Sampler
from qiskit_aer.primitives import Sampler as AerSampler

sampler = AerSampler()

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    loss=CrossEntropyLoss(),
    sampler=sampler
)

In [65]:
y_train = y_train.values
y_test = y_test.values

In [66]:
vqc.fit(X_train, y_train)

c:\Users\B_Jayanth\OneDrive\Desktop\QML\.venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [67]:
y_pred = vqc.predict(X_test)

In [68]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("QML Accuracy:", accuracy)

QML Accuracy: 0.7647058823529411


In [69]:
import numpy as np

np.save("qml_trained_weights.npy", vqc.weights)

In [70]:
import pickle

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# model Testing

In [71]:
import numpy as np
import pickle

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.utils.loss_functions import CrossEntropyLoss
from qiskit_aer.primitives import Sampler as AerSampler
from qiskit.algorithms.optimizers import COBYLA

In [72]:
loaded_weights = np.load("qml_trained_weights.npy")

In [73]:
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_aer.primitives import Sampler as AerSampler

feature_map = ZZFeatureMap(feature_dimension=3, reps=2)
ansatz = RealAmplitudes(num_qubits=3, reps=2)

sampler = AerSampler()

circuit = feature_map.compose(ansatz)

qnn = SamplerQNN(
    circuit=circuit,
    sampler=sampler,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

In [74]:
with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

In [75]:
new_sample = np.array([[10, 156.22621, 76263.53]])

In [76]:
new_sample_scaled = scaler.transform(new_sample)

c:\Users\B_Jayanth\OneDrive\Desktop\QML\.venv\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [77]:
output = qnn.forward(new_sample_scaled, loaded_weights)

prediction = np.argmax(output)

print("Predicted Label:", prediction)

ValueError: cannot reshape array of size 15 into shape (9,)